# Day 17: Methods, Properties & Encapsulation
## Writing Better Python Classes


---

## 1. Methods in a Class

### What is a method?

A **method** is a function that lives inside a class. It defines what an object can DO.

A regular function stands alone:

```python
def say_hello():
    print('Hello')
```

A method belongs to a class. It always has `self` as its first parameter:

```python
class Person:
    def greet(self):      # This is a METHOD
        print(f'Hello, I am {self.name}')
```

**Why put functions inside a class?** Because the action belongs with the data. A `Dog` should have `bark()`. A `BankAccount` should have `withdraw()`. The method and the data live together.

---

Python has **three kinds of methods**. We will learn each one separately, with a small example you can run right away.


### 1.1 Instance Methods

An **instance method** is the most common type. It works with `self` — the specific object that called the method.

**`self` means 'this particular object'.** When you write `dog1.bark()`, Python automatically sends `dog1` as `self`. So inside the method, `self.name` means `dog1.name`.

**Key points:**
- First parameter is always `self`
- No special decorator needed
- Use it when you need to read or change the object's own data


In [ ]:
# Instance Method — Simple Example
# Run this cell. See how each dog keeps its own data.

class Dog:
    def __init__(self, name, breed):
        self.name = name
        self.breed = breed

    def bark(self):              # <-- Instance method (has 'self')
        return f'{self.name} says Woof!'

    def introduce(self):         # <-- Another instance method
        return f'I am {self.name}, a {self.breed}.'

# Create two different dogs
dog1 = Dog('Bruno', 'Golden Retriever')
dog2 = Dog('Luna', 'Husky')

# Each dog behaves using ITS OWN data
print(dog1.bark())       # Bruno uses dog1's name
print(dog2.bark())       # Luna uses dog2's name
print(dog1.introduce())
print(dog2.introduce())



{'name': 'Bruno', 'breed': 'Golden Retriever'}


### 1.2 Class Methods

A **class method** works with the CLASS itself, not with individual objects.

Instead of `self`, it receives `cls` — which means 'the class.' When you change something through `cls`, it changes for ALL objects of that class.

**Key points:**
- First parameter is `cls` (not `self`)
- Must add `@classmethod` decorator on the line above
- Use it for things that belong to the whole class: counting objects, changing shared settings


In [ ]:
# Class Method — Simple Example
# Run this cell. The counter tracks ALL students, not just one.

class Student:
    total_students = 0          # Class variable — shared by everyone

    def __init__(self, name):
        self.name = name
        Student.total_students += 1    # Each new student adds 1

    @classmethod                 # <-- This makes it a class method
    def how_many(cls):           # <-- 'cls' = the Student class itself
        return f'There are {cls.total_students} students.'

# Create students
s1 = Student('Amit')
s2 = Student('Priya')
s3 = Student('Rahul')

# Call on the CLASS (preferred)
print(Student.how_many())

# Also works on an instance (Python figures it out)
print(s1.how_many())
print(s2.how_many())



{'name': 'Amit'}


### 1.3 Static Methods

A **static method** does NOT use `self` or `cls`. It is a regular function that we keep inside a class only because it is related to that class.

Think of it as a helper function that belongs with the class for organization.

**Key points:**
- No `self`, no `cls`
- Must add `@staticmethod` decorator on the line above
- Use it for utilities, validators, converters — anything related to the class but not tied to one object


In [3]:
# Static Method — Simple Example
# Run this cell. No object needed — call directly on the class.

class MathHelper:
    @staticmethod               # <-- This makes it a static method
    def is_even(number):        # <-- No 'self'! Just a normal parameter
        return number % 2 == 0

    @staticmethod
    def celsius_to_fahrenheit(celsius):
        return (celsius * 9/5) + 32

# Call directly on the CLASS — no object created
print(MathHelper.is_even(10))                # True
print(MathHelper.is_even(7))                 # False
print(MathHelper.celsius_to_fahrenheit(0))    # 32.0
print(MathHelper.celsius_to_fahrenheit(100))  # 212.0


True
False
32.0
212.0


### Quick Summary: The Three Method Types

| Type | First Parameter | Decorator | When to Use |
|------|----------------|-----------|-------------|
| **Instance** | `self` | *(none)* | Reading/writing THIS object's data |
| **Class** | `cls` | `@classmethod` | Working with data shared by ALL objects |
| **Static** | *(nothing)* | `@staticmethod` | Utility/helper function, no object data needed |

**Simple way to remember:**
- Instance method → works on ONE student's marks
- Class method → changes the school name for EVERYONE
- Static method → a calculator that checks pass/fail — it doesn't care which student


### Putting It Together: All Three in One Class

Now let us see all three types working together. Read each method and identify which type it is.


In [4]:
# All three method types in ONE class
# Read each method. Can you identify which is which?

class Student:
    school = 'Python Academy'       # Class variable

    def __init__(self, name, marks):
        self.name = name             # Instance variable
        self.marks = marks           # Instance variable

    # --- Instance Method (uses self) ---
    def get_grade(self):
        if self.marks >= 80:
            return 'A'
        elif self.marks >= 60:
            return 'B'
        else:
            return 'C'

    # --- Class Method (uses cls, has @classmethod) ---
    @classmethod
    def change_school(cls, new_name):
        cls.school = new_name        # Changes for ALL students

    # --- Static Method (no self, no cls, has @staticmethod) ---
    @staticmethod
    def is_passing(marks):
        return marks >= 40

# --- Test it ---
s1 = Student('Rahul', 85)

# Instance method
print('Grade:', s1.get_grade())

# Static method
print('Is 35 passing?', Student.is_passing(35))

# Class method
Student.change_school('Data Science Institute')
print('School:', s1.school)
print('School:', Student.school)


Grade: A
Is 35 passing? False
School: Data Science Institute
School: Data Science Institute


---

## 2. Properties — Controlling Attribute Access

### The Problem

Look at this code. Can you spot the danger?

```python
s = Student('Amit', 85)
s.marks = -9999    # Anyone can set ANY value!
```

There is nothing stopping someone from setting `marks` to `-9999` or `99999`. The data gets corrupted silently.

**The solution:** `@property` lets us CONTROL what happens when someone reads or writes an attribute. We can validate the value before storing it.


In [5]:
# The PROBLEM — no validation, no protection
# Run this. See how bad data gets in without any error.

class Student:
    def __init__(self, name, marks):
        self.name = name
        self.marks = marks     # Direct assignment — anything goes!

s = Student('Amit', 85)
print('Original marks:', s.marks)

s.marks = -9999                # This SHOULD be blocked, but it isn't!
print('After corruption:', s.marks)

s.marks = 'hello'              # Even this works!
print('After more corruption:', s.marks)


Original marks: 85
After corruption: -9999
After more corruption: hello


### How @property Works

A property is created in two parts:

**1. The Getter** — `@property` — called when you READ the value (`print(obj.marks)`)
**2. The Setter** — `@name.setter` — called when you WRITE the value (`obj.marks = 90`)

The actual data is stored in a private variable (like `self._marks`). The property acts as a gatekeeper.


In [8]:
# @property — Step 1: A READ-ONLY property (getter only)
# Can read, but cannot change directly.

class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property                     # <-- This makes 'area' a property
    def area(self):
        return 3.14159 * self.radius ** 2

    @property
    def circumference(self):
        return 2 * 3.14159 * self.radius

c = Circle(5)
print('Radius:', c.radius)
print('Area:', round(c.area, 2))           # No parentheses! c.area, not c.area()
print('Circumference:', round(c.circumference, 2))

# area and circumference update automatically when radius changes
c.radius = 10
print('New area:', round(c.area, 2))


Radius: 5
Area: 78.54
Circumference: 31.42
New area: 314.16


### Adding a Setter — Data Validation

A getter-only property is read-only. To allow writing (with validation), add a setter.

The setter is where you check the value and reject bad data.


In [9]:
# @property — Step 2: Getter + Setter with VALIDATION
# The setter blocks invalid values before they are stored.

class Student:
    def __init__(self, name, marks):
        self.name = name
        self._marks = 0          # Start with a default
        self.marks = marks       # This calls the SETTER below!

    @property                    # GETTER — runs when you READ obj.marks
    def marks(self):
        return self._marks

    @marks.setter                # SETTER — runs when you WRITE obj.marks = value
    def marks(self, value):
        if value < 0 or value > 100:
            raise ValueError(f'Marks must be 0-100. Got: {value}')
        self._marks = value

# Test with valid data
s = Student('Priya', 85)
print('Marks:', s.marks)

# Test with invalid data — the setter blocks it
try:
    s.marks = -500
except ValueError as e:
    print('Blocked:', e)

# Valid change works fine
s.marks = 92
print('Updated marks:', s.marks)


Marks: 85
Blocked: Marks must be 0-100. Got: -500
Updated marks: 92


### Computed Properties

A property does not have to store a value. It can CALCULATE a value from other attributes.

This is useful for things like `grade` (calculated from `marks`) or `full_name` (calculated from `first_name` + `last_name`).


In [ ]:
# Computed Property — grade is calculated from marks
# No setter needed because grade depends on marks.

class Student:
    def __init__(self, name, marks):
        self.name = name
        self._marks = 0
        self.marks = marks

    @property
    def marks(self):
        return self._marks

    @marks.setter
    def marks(self, value):
        if value < 0 or value > 100:
            raise ValueError(f'Marks must be 0-100. Got: {value}')
        self._marks = value

    @property                    # Computed — no setter, depends on marks
    def grade(self):
        if self.marks >= 80:
            return 'A'
        elif self.marks >= 60:
            return 'B'
        elif self.marks >= 40:
            return 'C'
        else:
            return 'F'

s = Student('Amit', 85)
print(f'{s.name}: Marks={s.marks}, Grade={s.grade}')

s.marks = 55
print(f'{s.name}: Marks={s.marks}, Grade={s.grade}')


---

## 3. Encapsulation — Protecting Your Data

### What is Encapsulation?

Encapsulation means two things:

1. **Bundle** data (attributes) and methods together inside a class
2. **Restrict** direct access to internal details

Think of it like this: When you use a TV remote, you press the 'volume up' BUTTON. You do not open the remote and touch the wires inside. The button is the public interface. The wires are private internals.

Encapsulation is about giving users of your class a clean, safe way to interact with it — and hiding the complexity inside.


### Three Levels of Access in Python

| Prefix | Name | Meaning | Protection |
|--------|------|---------|------------|
| `name` | **Public** | Anyone can access freely | None |
| `_name` | **Protected** | Please don't touch this | Convention — IDEs warn you |
| `__name` | **Private** | Stay out | Python renames it to prevent accidents |

Python's philosophy is 'we are all adults here.' Even private attributes can be accessed if you really try. The goal is to **prevent accidents**, not to enforce security.


In [ ]:
# Public attribute — anyone can read and write
# No restrictions at all.

class Person:
    def __init__(self, name):
        self.name = name       # Public — use it freely

p = Person('Amit')
print(p.name)       # Reading — fine
p.name = 'Rahul'    # Writing — fine
print(p.name)


In [6]:
# Protected attribute — convention says 'don't touch'
# The _ prefix means 'this is internal, please don't use it directly.'

class Person:
    def __init__(self, name, age):
        self.name = name        # Public
        self._age = age         # Protected — _ prefix is a warning

p = Person('Amit', 25)
print(p._age)        # It still works, but your IDE will show a warning
p._age = 30          # This also works, but you are ignoring the convention
print(p._age)   
# The _ prefix says: you should not touch this attribute.


25
30


In [12]:
# Private attribute — Python RENAMES it to block access
# The __ prefix triggers name mangling.

class Person:
    def __init__(self, name, id_number):
        self.name = name           # Public
        self.__id = id_number      # Private — __ prefix

p = Person('Amit', 'ID12345')

print(p.name)          # Works fine

# This will FAIL:
try:
    print(p.__id)
except AttributeError as e:
    print('Cannot access __id directly!')
    print(f'Error: {e}')

# Python secretly renamed __id to _Person__id
# You CAN access it if you know the mangled name (but you should not)
print('Mangled name works:', p._Person__id)


Amit
Cannot access __id directly!
Error: 'Person' object has no attribute '__id'
Mangled name works: ID12345


### Real-World Example: BankAccount

This class uses everything we have learned:
- **Private attribute** (`__balance`) — cannot be changed directly
- **Read-only property** (`balance`) — can view, cannot set
- **Controlled methods** (`deposit`, `withdraw`) — the ONLY way to change balance

Run the cell and try to break it. You will find it is hard to use this class incorrectly.


In [ ]:
# BankAccount — Full encapsulation example
# __balance is private. Only deposit() and withdraw() can change it.

class BankAccount:
    def __init__(self, holder, balance):
        self.holder = holder
        self.__balance = balance       # PRIVATE

    @property
    def balance(self):                 # READ-ONLY property
        return self.__balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError('Deposit amount must be positive')
        self.__balance += amount
        return f'Deposited Rs.{amount}. New balance: Rs.{self.__balance}'

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError('Withdrawal amount must be positive')
        if amount > self.__balance:
            raise ValueError(f'Insufficient funds! Balance is Rs.{self.__balance}')
        self.__balance -= amount
        return f'Withdrew Rs.{amount}. New balance: Rs.{self.__balance}'

# --- Test it ---
account = BankAccount('Rahul', 5000)
print(account.deposit(2000))
print(account.withdraw(1500))

# Cannot access __balance directly
try:
    print(account.__balance)
except AttributeError:
    print('Direct access to __balance blocked!')
    print('Use account.balance property instead:', account.balance)


---

## 4. Name Mangling — What Happens to `__private`

### The Mechanism

When you write `self.__secret = 'hidden'`, Python automatically renames it to `self._ClassName__secret`.

This is called **name mangling**. The `__` prefix tells Python: 'Rename this so nobody accidentally uses the wrong name.'

You can see the mangled names using `obj.__dict__`.


In [ ]:
# Name Mangling — See it in action
# Look at the __dict__ output carefully.

class Person:
    def __init__(self, name):
        self.name = name            # Normal attribute
        self.__id = 'ID12345'       # Will be renamed!

p = Person('Amit')

# Look at ALL attributes stored on the object
print(p.__dict__)
# Output: {'name': 'Amit', '_Person__id': 'ID12345'}
#                           ^^^^^^^^^^^^^^
#                           __id became _Person__id !

# So this fails:
try:
    print(p.__id)
except AttributeError as e:
    print(f'Cannot access __id: {e}')

# But this works (though you should NOT use it):
print('Mangled access:', p._Person__id)


### Why Name Mangling Exists

The real reason for name mangling is **inheritance**. A child class might accidentally use the same `__name` as the parent. Name mangling prevents them from conflicting.

Each class gets its own mangled name:
- `Parent.__secret` becomes `_Parent__secret`
- `Child.__secret` becomes `_Child__secret`

These are TWO DIFFERENT variables. They do not overwrite each other.


In [13]:
# Name Mangling in Inheritance
# Parent and Child each have their own __secret — they do not clash.

class Parent:
    def __init__(self):
        self.__secret = 'hidden'      # Becomes _Parent__secret

class Child(Parent):
    def __init__(self):
        super().__init__()
        self.__secret = 'child secret'  # Becomes _Child__secret (DIFFERENT!)

c = Child()
# Child has TWO separate attributes:
print(c.__dict__)
# {'_Parent__secret': 'hidden', '_Child__secret': 'child secret'}
#  ^^^^^^^^^^^^^^^^^           ^^^^^^^^^^^^^^^^^
#  Parent's __secret           Child's __secret — no conflict!

print("Parent's secret:", c._Parent__secret)
print("Child's secret:", c._Child__secret)


{'_Parent__secret': 'hidden', '_Child__secret': 'child secret'}
Parent's secret: hidden
Child's secret: child secret


---

## Practice Exercises

Try these on your own. Each exercise uses concepts from this lesson.

---

**Exercise 1: Temperature Class**

Create a `Temperature` class with:
- A `celsius` property with a setter that prevents values below -273.15 (absolute zero)
- A `fahrenheit` property that computes from celsius (getter only)

**Exercise 2: Product Class**

Create a `Product` class with:
- A private `__price` attribute
- A `price` property with a setter that rejects negative values
- A `discount_price` computed property that returns 10% off the price

**Exercise 3: Static Validator**

Add a `@staticmethod` called `is_valid_email(email)` to any class. It should check that the email contains `@` and a `.` after the `@`.


---

## What We Learned Today

| Concept | How to Spot It | What It Does |
|---------|---------------|--------------|
| **Instance Method** | First parameter is `self` | Works with one object's data |
| **Class Method** | Has `@classmethod`, first parameter is `cls` | Works with class-level data |
| **Static Method** | Has `@staticmethod`, no `self` or `cls` | Utility function in a class |
| **@property** | `@property` above a method | Controls reading of an attribute |
| **@name.setter** | `@propname.setter` above a method | Controls writing to an attribute |
| **_protected** | Single underscore `_name` | Please don't touch — convention |
| **__private** | Double underscore `__name` | Name mangled — hard to access accidentally |
| **Encapsulation** | Private data + public methods + properties | Safe, controlled access to data |

---

**In Day 18:** Magic Methods — making your objects work with `+`, `print()`, `len()`, and more.
